## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

Set up the working environment:

In [ ]:
# To ensure reproducibility
seed = 42

# Load basic libraries
import os
import sys
from pathlib import Path

# Buscamos la raíz del proyecto para añadirla al path y poder importar 'src'
root_path = Path(os.getcwd()).resolve().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

# Importamos la configuración centralizada desde src/
import src.config as config

# Usamos las rutas predefinidas
file_path = config.CLINICAL_DATA_PATH
results_path = config.get_results_path("whole_data")

print(f"Project's root: {root_path}")
print(f"Loading data from: {file_path}")
print(f"Saving results to: {results_path}")

Load libraries:

In [ ]:
# DATA WRANGLING AND STATISTICS
import pandas as pd
import numpy as np
from scipy.stats import randint, uniform, loguniform

# DATA PREPROCESSING
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer

# HYPERPARAMETER TUNING
import optuna
from optuna.integration import OptunaSearchCV
from optuna.distributions import IntDistribution, FloatDistribution, CategoricalDistribution
optuna.logging.set_verbosity(optuna.logging.WARNING)

# MODEL TRAINING 
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# MODEL EVALUATION
from sklearn.metrics import make_scorer, recall_score

# CUSTOM MODULES
from src.models import *


## Data

Load the data and check the structure: and split the data into target class and features:

In [ ]:
# Load data
df = pd.read_parquet(file_path)

# Check general structure
df.info()

Separate the features from the target class:

In [ ]:
# Drop the target class and the non-informative features
X = df.drop(["AF_recurrence"], axis=1)

# Select the target class and encode it manually
y = df["AF_recurrence"].map({"no":0, "yes":1})

Divide data set into train and test set:

In [ ]:
# Divide into train and test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=seed,
    shuffle=True,
    stratify=y,
    )

# Compute the predicted class ratio
negatives = (y_train == 0).sum()
positives = (y_train == 1).sum()

ratio = negatives / positives

print(f"Negative cases: {negatives}, Positive cases: {positives}")
print(f"Imbalance ratio suggested: {ratio:.2f}")

## Training and optimization

Training each model with stratified 5-fold cross validation:

In [ ]:
# Number of splits
n = 5

# Cross-validation strategy
my_cv = StratifiedKFold(n_splits=n, shuffle=True, random_state=seed)

Define metrics to evaluate:

In [ ]:
# Define new metrics
specificity_score = make_scorer(recall_score, pos_label=0)

# Set up the scoring dictionary for cross-validation
scoring_dict = {
        'Accuracy': 'accuracy',
        'Precision': 'precision',
        'Recall': 'recall',
        'Specificity': specificity_score,
        'F1-Score': 'f1',
        'ROC-AUC': 'roc_auc',
        'PR-AUC': 'average_precision'
    }

#### Logistic Regression (Elastic Net)

Define the parameter distributions and set up the pipeline:

In [ ]:
# Hyperparameters search space
params_EN = {
    'clf__l1_ratio': FloatDistribution(0, 1),
    'clf__C': FloatDistribution(1e-4, 1e3, log=True)
    }

# Get the preprocessor
preprocessor_EN = get_regres_preprocessor(X_train)

# Full pipeline
pipe_EN = Pipeline(steps=[
    ('preprocessor', preprocessor_EN),
    ('clf', LogisticRegression(random_state=seed,
                            solver='saga',
                            max_iter=10000))
])

Train and optimize the model:

In [ ]:
(optimized_EN, 
cv_results_EN,
tpr_EN, fpr_EN,
precs_EN, recs_EN,
study_EN) = optimize_model_optuna_search(pipe_EN, params_EN, 
                                                X_train, y_train, X_test, y_test, 
                                                metrics_dict=scoring_dict,
                                                cv=my_cv, 
                                                seed=seed)

In [ ]:
# Display and save the optimization history
study_EN.set_user_attr("model_name", "Elastic Net")
plot_optimization_history(study_EN, output_dir=results_path)

Extract hyerparameters and display them to check if the distribution is wide enough:

In [ ]:
save_model(optimized_EN, output_dir=results_path)

Check the confusion matrix:

In [ ]:
# Compute predictions for the test set
y_pred_EN = optimized_EN.predict(X_test)

# Display confusion matrix
plot_confusion_matrix(y_true=y_test, y_pred=y_pred_EN, 
        title="Elastic Net", output_dir=results_path, file_prefix="EN")

Check the overfitting:

In [ ]:
plot_overfitting_bars(cv_results_EN, model_name="EN", output_dir=results_path)

### Support Vector Machine


Set up the pipeline and the parameter distributions:

In [ ]:
# Hyperparameters search space
params_dist_SVM = {
    'clf__C': FloatDistribution(1e-5, 1e4, log=True),
    'clf__kernel': CategoricalDistribution(['linear', 'rbf', 'poly', 'sigmoid']),
    'clf__gamma': CategoricalDistribution(['scale', 'auto']),
    'clf__degree': IntDistribution(2, 6),
    'clf__class_weight': CategoricalDistribution([None, 'balanced'])
}

# Get preprocessor
preprocessor_SVM = get_geom_preprocessor(X_train)

# Full pipeline
pipe_SVM = Pipeline(steps=[
    ('preprocessor', preprocessor_SVM),
    ('clf', SVC(random_state=seed))
])

Train and optimize the model:

In [ ]:
(optimized_SVM, 
cv_results_SVM,
tpr_SVM, fpr_SVM,
precs_SVM, recs_SVM,
study_SVM) = optimize_model_optuna_search(pipe_SVM, params_dist_SVM, 
                                    X_train, y_train, X_test, y_test, 
                                    metrics_dict=scoring_dict,
                                    cv=my_cv)

In [ ]:
# Display and save the optimization history
study_SVM.set_user_attr("model_name", "SVM")
plot_optimization_history(study_SVM, output_dir=results_path)

Show hyperparameters to check if the distributions are wide enough:

In [ ]:
save_model(optimized_SVM, results_path)

Check the confusion matrix:

In [ ]:
# Compute predictions for the test set
y_pred_SVM = optimized_SVM.predict(X_test)

# Display confusion matrix
plot_confusion_matrix(y_true=y_test, y_pred=y_pred_SVM, 
        title="SVM", output_dir=results_path, file_prefix="SVM")

Check the overfitting:

In [ ]:
plot_overfitting_bars(cv_results_SVM, model_name="SVM", output_dir=results_path)

### Random Forest

Set up the pipeline and the parameters distributions:

In [ ]:
# Hyperparameters search space
params_dist_RF = {
    'clf__n_estimators': IntDistribution(20, 300),
    'clf__max_depth': IntDistribution(2, 32),
    'clf__max_features': CategoricalDistribution(['sqrt', 'log2', None]),
    'clf__min_samples_split': IntDistribution(2, 20),
    'clf__criterion': CategoricalDistribution(['gini', 'entropy']),
    'clf__class_weight': CategoricalDistribution([None, 'balanced', 'balanced_subsample'])
}

# Get the preprocessor
preprocessor_RF = get_bagg_preprocessor(X_train)

# Full pipeline
pipe_RF = Pipeline(steps=[
    ('preprocessor', preprocessor_RF),
    ('clf', RandomForestClassifier(random_state=seed))
])

Train and optimize the model:

In [ ]:
(optimized_RF, 
cv_results_RF,
tpr_RF, fpr_RF,
precs_RF, recs_RF,
study_RF) = optimize_model_optuna_search(pipe_RF, params_dist_RF, 
                                                X_train, y_train, X_test, y_test, 
                                                metrics_dict=scoring_dict,
                                                cv=my_cv)

In [ ]:
# Display and save the optimization history
study_RF.set_user_attr("model_name", "Random Forest")
plot_optimization_history(study_RF, output_dir=results_path)

Show hyperparameters to check if the distributions are wide enough:

In [ ]:
save_model(optimized_RF, results_path)

Check the confusion matrix:

In [ ]:
# Compute predictions for the test set
y_pred_RF = optimized_RF.predict(X_test)

# Display confusion matrix
plot_confusion_matrix(y_true=y_test, y_pred=y_pred_RF, 
        title="Random Forest", output_dir=results_path, file_prefix="RF")

Check the overfitting:

In [ ]:
plot_overfitting_bars(cv_results_RF, model_name="RF", output_dir=results_path)

### Extreme Gradient Boosting

Set up the pipeline and the parameters distributions:

In [ ]:
# Hyperparameters search space
params_dist_XGB = {
    'clf__n_estimators': IntDistribution(20, 300),
    'clf__max_depth': IntDistribution(3, 10),
    'clf__learning_rate': FloatDistribution(0.01, 0.3),
    
    'clf__subsample': FloatDistribution(0.4, 0.6),
    'clf__colsample_bytree': FloatDistribution(0.4, 0.6),
    
    # Regularization parameters:
    'clf__reg_alpha': FloatDistribution(0, 10),
    'clf__reg_lambda': FloatDistribution(1, 10)
}

# Get the preprocessor
preprocessor_XGB = get_boost_preprocessor(X_train)

# Full pipeline
pipe_XGB = Pipeline(steps=[
    ('preprocessor', preprocessor_XGB),
    ('clf', XGBClassifier(random_state=seed,
                        scale_pos_weight=ratio, 
                        eval_metric='logloss'))
])

Train and optimize the model:

In [ ]:
(optimized_XGB, 
cv_results_XGB,
tpr_XGB, fpr_XGB,
precs_XGB, recs_XGB,
study_XGB) = optimize_model_optuna_search(pipe_XGB, params_dist_XGB,
                                                X_train, y_train, X_test, y_test,
                                                metrics_dict=scoring_dict,
                                                cv=my_cv,
                                                seed=seed)

In [ ]:
# Display and save the optimization history
study_XGB.set_user_attr("model_name", "XGBoost")
plot_optimization_history(study_XGB, output_dir=results_path)

Show hyperparameters to check if the distributions are wide enough:

In [ ]:
save_model(optimized_XGB, results_path)

Check the confusion matrix:

In [ ]:
# Compute predictions for the test set
y_pred_XGB = optimized_XGB.predict(X_test)

# Display confusion matrix
plot_confusion_matrix(y_true=y_test, y_pred=y_pred_XGB, 
        title="XGBoost", output_dir=results_path, file_prefix="XGB")

Check the overfitting:

In [ ]:
plot_overfitting_bars(cv_results_XGB, model_name="XGB", output_dir=results_path)

## Save results

In [ ]:
models = ["Elastic Net", "SVM", "Random Forest", "XGBoost"]

models_dict = {
    "Elastic Net": cv_results_EN,
    "SVM": cv_results_SVM,
    "Random Forest": cv_results_RF,
    "XGBoost": cv_results_XGB
}

### Main metrics

In [ ]:
results = save_metrics_results(models_dict=models_dict, output_dir=results_path)

results.head()

In [ ]:
plot_metrics_bars(results, 
                metrics=['Accuracy', 'Precision', 'Recall', 'Specificity', 'ROC-AUC', 'PR-AUC'],
                output_dir=results_path)

### ROC and PR curves

Join the metrics into a csv file:

In [ ]:
# Save the false/true positive rates values into a csv
fpr = [fpr_EN, fpr_SVM, fpr_RF, fpr_XGB]
tpr = [tpr_EN, tpr_SVM, tpr_RF, tpr_XGB]

roc_results = save_curves_results(models, fpr, tpr, curve_type='roc', 
                                output_dir=results_path)

# Save the precision and recall values for the PR curves into a csv
precs = [precs_EN, precs_SVM, precs_RF, precs_XGB]
recs = [recs_EN, recs_SVM, recs_RF, recs_XGB]

pr_results = save_curves_results(models, recs, precs, curve_type='pr', 
                                output_dir=results_path)

Plot ROC curves:

In [ ]:
plot_model_curves(roc_results, 
                x_col='False Positive Rate', y_col='True Positive Rate', 
                curve_type='roc', title="ROC curves",
                output_dir=results_path)

Plot PR curves:

In [ ]:
plot_model_curves(pr_results, 
                x_col='Recall', y_col='Precision', 
                curve_type='pr', title="Precision-Recall curves",
                output_dir=results_path)